In [1]:
import yaml

In [2]:
%%writefile /kaggle/working/dataset.yaml
path: /kaggle/input/datasets/ramazan2020/abdomen-acikveri/det_data/fold0
train: images/train
val: images/val
names:
  0: acute_cholecystitis
  1: kidney_ureter_stone
  2: acute_pancreatitis
  3: aortic_aneurysm_dissection
  4: acute_appendicitis
  5: acute_diverticulitis


Writing /kaggle/working/dataset.yaml


In [5]:
import os, sys
from pathlib import Path


fold = 0
fold_dir = Path("/kaggle/input/datasets/ramazan2020/abdomen-acikveri/det_data/fold0")
DET_DATA_DIR=Path("/kaggle/input/datasets/ramazan2020/abdomen-acikveri/det_data")
dataset_yaml = Path("/kaggle/working/dataset.yaml")
print("YOLO veri seti:", fold_dir, "→ var?", fold_dir.exists())
print("dataset.yaml :", dataset_yaml.exists())

YOLO veri seti: /kaggle/input/datasets/ramazan2020/abdomen-acikveri/det_data/fold0 → var? True
dataset.yaml : True


In [6]:
import torch
print(f"PyTorch         : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM (GB)       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
elif torch.backends.mps.is_available():
    print("MPS device      : Apple Silicon")
else:
    print("⚠️  GPU yok — Kaggle/Colab GPU runtime önerilir.")

PyTorch         : 2.10.0+cpu
CUDA available  : False
⚠️  GPU yok — Kaggle/Colab GPU runtime önerilir.


In [7]:
from dataclasses import dataclass, field


In [8]:
@dataclass
class DetConfig:
    model: str = "yolov8x.pt"
    img_size: int = 512
    batch_size: int = 16
    epochs: int = 50
    mosaic: float = 0.5
    mixup: float = 0.15
    workers_mps: int = 2
    workers_cuda: int = 8
    workers_cpu: int = 4
    patience: int = 20


In [9]:
DEFAULT_DET = DetConfig()


In [10]:
from dataclasses import replace
fast_cfg = replace(DEFAULT_DET,
    model="yolov8s.pt",
    epochs=10,
    img_size=512,
    batch_size=16,
    patience=5,
)
print("Eğitim ayarları:")
for k, v in fast_cfg.__dict__.items():
    print(f"  {k}: {v}")

Eğitim ayarları:
  model: yolov8s.pt
  img_size: 512
  batch_size: 16
  epochs: 10
  mosaic: 0.5
  mixup: 0.15
  workers_mps: 2
  workers_cuda: 8
  workers_cpu: 4
  patience: 5


In [ ]:

from __future__ import annotations

import argparse
import multiprocessing
import os
import platform
import shutil
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm



# ---------------------------------------------------------------------------
# EĞİTİM
# ---------------------------------------------------------------------------
def train_yolo(fold: int, cfg=DEFAULT_DET, project: str = "runs/det") -> Path:
    import torch
    from ultralytics import YOLO

    fold_dir = DET_DATA_DIR / f"fold{fold}"

    # Label'ı olan ama görüntüsü eksik dosyaları tespit et
    missing = []
    for split in ("train", "val"):
        for lbl in (fold_dir / "labels" / split).glob("*.txt"):
            img = fold_dir / "images" / split / (lbl.stem + ".png")
            if not img.exists():
                missing.append(img)
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} görüntü eksik (export sırasında cv2.imwrite başarısız olmuş). "
            f"İlk örnek: {missing[0]}\n"
            "export_yolo_dataset() tekrar çalıştırılmalı."
        )

    # MPS → CUDA → CPU öncelik sırası
    if torch.backends.mps.is_available():
        device = "mps"
        n_workers = cfg.workers_mps
    elif torch.cuda.is_available():
        device = "0"
        n_workers = min(cfg.workers_cuda, os.cpu_count() or 4)
    else:
        device = "cpu"
        n_workers = min(cfg.workers_cpu, os.cpu_count() or 2)

    model = YOLO(cfg.model)
    run_name = f"fold{fold}_{Path(cfg.model).stem}"
    model.train(
        data=str(dataset_yaml),
        imgsz=cfg.img_size,
        epochs=cfg.epochs,
        batch=cfg.batch_size,
        mosaic=cfg.mosaic,
        mixup=cfg.mixup,
        project=project,
        name=run_name,
        seed=42,
        deterministic=False,
        close_mosaic=10,
        patience=cfg.patience,
        device=device,
        workers=n_workers,
        cache=False,
        amp=True,
    )
    # Ultralytics exist_ok=False ile run ismini otomatik suffix'ler (fold0_yolov8x → fold0_yolov8x2 vb.)
    # Bu nedenle sabit run_name yerine trainer'ın gerçek save_dir'ini kullanıyoruz.
    return Path(model.trainer.save_dir) / "weights" / "best.pt"


# ---------------------------------------------------------------------------
# İNFERANS + 3B POST-PROCESSING
# ---------------------------------------------------------------------------
def predict_volume(weights: Path,
                   case_dir: Path,
                   conf: float = 0.2,
                   min_slice_run: int = 3) -> pd.DataFrame:
    """
    Bir vakadaki tüm kesitlerde YOLO çıkarımı yapar ve **sadece ardışık
    `min_slice_run` kesit boyunca süregelen tespitleri** tutar.
    Bu, bağımsız tek-kesitlik false-positive'leri büyük ölçüde süzer.
    """
    from ultralytics import YOLO
    model = YOLO(str(weights))
    dcm_paths = sorted(p for p in case_dir.iterdir() if p.suffix.lower() == ".dcm")

    all_rows = []
    for p in tqdm(dcm_paths, desc=f"predict {case_dir.name}"):
        ds = read_dicom(p)
        hu = dicom_to_hu(ds)
        img = (hu_to_three_channel(hu, DEFAULT_WINDOWS) * 255).astype(np.uint8)
        res = model.predict(img, conf=conf, verbose=False)[0]
        for box, score, cls in zip(res.boxes.xyxy.cpu().numpy(),
                                   res.boxes.conf.cpu().numpy(),
                                   res.boxes.cls.cpu().numpy()):
            all_rows.append({
                "case": case_dir.name,
                "image_id": int(p.stem),
                "class": int(cls),
                "score": float(score),
                "x1": float(box[0]), "y1": float(box[1]),
                "x2": float(box[2]), "y2": float(box[3]),
            })

    df = pd.DataFrame(all_rows)
    if df.empty or min_slice_run <= 1:
        return df

    # Ardışık kesit süreklilik filtresi (sınıf başına IoU tabanlı eşleştirme)
    filt = []
    df = df.sort_values(["class", "image_id"]).reset_index(drop=True)
    for cls, grp in df.groupby("class"):
        grp = grp.reset_index(drop=True)
        kept = [False] * len(grp)
        for i, row in grp.iterrows():
            run = 1
            j = i + 1
            while j < len(grp) and grp.iloc[j]["image_id"] - grp.iloc[j - 1]["image_id"] <= 2:
                if _iou(row, grp.iloc[j]) > 0.3:
                    run += 1
                    j += 1
                else:
                    break
            if run >= min_slice_run:
                for k in range(i, i + run):
                    kept[k] = True
        filt.append(grp[kept])
    return pd.concat(filt, ignore_index=True) if filt else df.iloc[0:0]


def _iou(a, b) -> float:
    ax1, ay1, ax2, ay2 = a["x1"], a["y1"], a["x2"], a["y2"]
    bx1, by1, bx2, by2 = b["x1"], b["y1"], b["x2"], b["y2"]
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = max(0.0, (ax2 - ax1) * (ay2 - ay1))
    ub = max(0.0, (bx2 - bx1) * (by2 - by1))
    return inter / max(ua + ub - inter, 1e-6)




In [16]:
DIR = Path("/kaggle/working/runs/det")

DIR.mkdir(parents=True, exist_ok=True)


In [17]:
%%capture install_log
!pip install -q ultralytics pydicom opencv-python-headless tqdm pandas openpyxl scikit-learn pyyaml


In [ ]:
weights_path = train_yolo(fold=fold, cfg=fast_cfg, project=str(DIR))
print("En iyi ağırlık:", weights_path)

Ultralytics 8.4.52 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=False, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=fold0_yolov8s-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=5, p